# Доп. главы методов оптимизации — домашнее задание № 1

**По материалу занятий 3, 4 и 5:**

* **Занятие 3.** Стохастическая оптимизация: постановка $\min_x f(x)=\mathbb E_{\xi\sim\mathcal D} f(x,\xi)$,
  online/offline, finite-sum; SGD; предположения (выпуклость, $L$-гладкость, несмещённость, ограниченная дисперсия,
  affine noise, тяжёлые хвосты); теорема о сходимости SGD для $\mu$-сильно выпуклых функций; тюнинг шага (Stich, 2019)
  и оценка $\widetilde O\!\left(\exp(-K\mu/L)R_0^2+\sigma^2/(\mu^2K)\right)$; семинар: high-probability, клиппинг,
  нормализация, выбор $\gamma_k$, сходимость по последней точке.
* **Занятие 4.** Субградиентный метод: субградиент и субдифференциал, $M$-липшицевость $\Leftrightarrow$
  ограниченность субградиентов, оценка $O(MR_0/\sqrt K)$, оптимальный шаг $\gamma=R_0/(M\sqrt K)$; адаптивные методы:
  AdaGrad-Norm, AdaGrad, RMSProp, Adam, AMSGrad, AdamW; убывающие шаги.
* **Занятие 5.** Parameter-free оптимизация: adaptive против parameter-free; наивный подход — Bisection
  (неподвижная точка $\gamma=\phi(\gamma)$, стоимость $O(\log\log(\gamma_{hi}/\gamma_{lo}))$);
  продвинутый подход — DoG, D-Adaptation, Prodigy; sign-SGD, связь с Adam, $L_\infty$-гладкость,
  контрпримеры и ALIAS Sign-SGD.

---

Это домашнее задание: пять экспериментов, каждый проверяет численно одно конкретное
утверждение из лекций. Теоретические задачи в это задание не входят.

### Правила

* Ноутбук должен выполняться сверху вниз без правок (`Kernel → Restart & Run All`).
* Пишется **только код методов**. Отрисовка и печать таблиц делаются готовыми функциями
  `plot_curves`, `plot_sweep`, `show_table` из раздела «Общий каркас» — свои варианты
  оформления графиков писать не нужно и не следует.
* Каждый эксперимент заканчивается **письменным выводом** (2–5 предложений). Вывод
  «график построен» — это не вывод; нужно сравнение численного результата с теоретическим
  предсказанием из лекции.
* Все запуски — с фиксированным `seed`; там, где в формуле стоит $\mathbb E$, в коде должно быть
  усреднение по нескольким независимым запускам.
* Все методы внутри одного эксперимента стартуют из **одной и той же** точки $x^0$ — она задана
  в ноутбуке константой; менять её нельзя, иначе кривые несравнимы.
* Разрешены `numpy`, `matplotlib`, `scipy`. Готовые оптимизаторы (`torch.optim`, `sklearn`) использовать
  **нельзя** — все методы реализуются вручную; `scipy.optimize` разрешён только для вычисления $f^*$.

### Баллы

| | Задача | Баллы |
|---|---|---|
| Э1 | Плато SGD: роль шага и размера батча | 10 |
| Э2 | Расписания шага и скорость $1/K$ | 10 |
| Э3 | Субградиентный метод на $\|Ax-b\|_1$ | 10 |
| Э4 | Адаптивные методы; расходимость Adam | 10 |
| Э5 | Parameter-free: Bisection, DoG, D-Adaptation, Prodigy, sign-SGD | 10 |
| | **Итого** | **50** |

### Обозначения

$x^*\in\arg\min f$, $f^*=f(x^*)$, $R_0=\|x_0-x^*\|$, $\|\cdot\|=\|\cdot\|_2$;
$\mu$ и $L$ — константы сильной выпуклости и гладкости, $M$ — константа липшицевости,
$\sigma^2$ — дисперсия стохастического градиента, $K$ (или $T$) — число итераций.

## Настройка окружения

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (8.0, 4.6),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "lines.linewidth": 1.9,
    "legend.framealpha": 0.9,
})

SEED = 0
np.set_printoptions(precision=4, suppress=True)
print("numpy", np.__version__)

## Общий каркас

Ниже — утилиты, которые **писать не нужно**: они выдаются готовыми, чтобы у всех совпадали
и формат кода, и вид графиков. Ваша задача — реализовать сами методы; отрисовка и печать таблиц
делаются вызовом этих функций.

**Соглашения, которых нужно придерживаться во всех задачах:**

1. Метод — это функция, которая возвращает **историю** в виде одномерного массива
   (значения $f(x_k)-f^*$, $\|x_k-x^*\|^2$ и т.п.), начиная **с точки $x^0$**: `hist[0]` — значение
   в стартовой точке. Тогда все кривые на графике заведомо выходят из одной точки.
2. Все методы в пределах одного эксперимента стартуют из **одного и того же** $x^0$
   (для каждой задачи он задан в этом ноутбуке как константа).
3. График сходимости строится **только** через `plot_curves`, график «результат против параметра» —
   через `plot_sweep`. В обоих логарифм **только по оси $y$**, ось $x$ линейная; в `plot_sweep`
   параметр отложен в декадах ($\log_{10}$ значения), поэтому узлы сетки не слипаются.
4. Числовые сводки печатаются через `show_table`.

In [ ]:
EPS_PLOT = 1e-16          # пол для логарифмической оси
PALETTE = plt.cm.tab10(np.arange(10))


def best_so_far(v):
    '''min_{i<=k} v_i — «лучшее значение к моменту k».'''
    return np.minimum.accumulate(np.asarray(v, dtype=float))


def plot_curves(curves, xlabel, ylabel, title, ref=None, x=None, ax=None,
                figsize=(8.2, 4.6), legend_fs=9, mark_start=True):
    '''Единый формат графика сходимости: логарифм по y, ЛИНЕЙНАЯ ось x.

    curves : {подпись: 1-D массив}, hist[0] — значение в стартовой точке
    ref    : (подпись, массив) — эталонная кривая, чёрный пунктир
    x      : ось абсцисс (по умолчанию 0, 1, 2, ...)
    '''
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    for (name, y), color in zip(curves.items(), PALETTE):
        y = np.maximum(np.asarray(y, dtype=float), EPS_PLOT)
        ax.plot(np.arange(len(y)) if x is None else x, y, color=color, label=name)
    if ref is not None:
        y = np.maximum(np.asarray(ref[1], dtype=float), EPS_PLOT)
        ax.plot(np.arange(len(y)) if x is None else x, y, "k--", lw=1.2, label=ref[0])
    starts = {float(np.asarray(y, dtype=float)[0]) for y in curves.values()}
    if mark_start and len(starts) == 1:
        x0 = 0 if x is None else np.asarray(x)[0]
        ax.plot([x0], [max(starts.pop(), EPS_PLOT)], "o", color="black", ms=6, zorder=5,
                label="общая стартовая точка")
    ax.set_yscale("log")
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=legend_fs)
    if own:
        plt.tight_layout(); plt.show()
    return ax


def decades(v):
    '''Параметр -> его десятичный логарифм: ось x у нас всегда ЛИНЕЙНАЯ,
    поэтому параметр, пробегающий порядки величины, откладывается как log10.'''
    return np.log10(np.asarray(v, dtype=float))


def plot_sweep(x, series, xlabel, ylabel, title, hlines=None, vlines=None,
               ax=None, figsize=(8.2, 4.6), legend_fs=9):
    '''График «итог против параметра». Логарифм только по оси y; ось x линейная,
    а сам параметр отложен в декадах (log10), иначе точки сетки сливаются у нуля.

    series : {подпись: массив той же длины, что x}
    hlines : {подпись: уровень}, vlines : {подпись: значение параметра}
    '''
    own = ax is None
    if own:
        _, ax = plt.subplots(figsize=figsize)
    for (name, y), color in zip(series.items(), PALETTE):
        ax.plot(decades(x), np.maximum(np.asarray(y, dtype=float), EPS_PLOT),
                "o-", color=color, label=name)
    for name, lvl in (hlines or {}).items():
        ax.axhline(lvl, color="k", ls="--", lw=1.2, label=name)
    for name, pos in (vlines or {}).items():
        ax.axvline(decades(pos), color="grey", ls=":", lw=1.2, label=name)
    ax.set_yscale("log")
    ax.set_xlabel(rf"$\log_{{10}}$ {xlabel}"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=legend_fs)
    if own:
        plt.tight_layout(); plt.show()
    return ax


def show_table(headers, rows, width=15):
    '''Печать сводки в едином формате: float — в экспоненциальной записи.'''
    def cell(v):
        return f"{v:.3e}" if isinstance(v, float) else str(v)
    head = " | ".join(str(h).rjust(width) for h in headers)
    print(head); print("-" * len(head))
    for r in rows:
        print(" | ".join(cell(v).rjust(width) for v in r))

### Квадратичная задача (Э1, Э2)

$$f(x)=\tfrac12 x^{\!\top}Ax-\langle b,x\rangle,\qquad A=A^{\!\top}\succ0,\quad
\operatorname{spec}(A)\subset[\mu,L],$$
стохастический оракул $\nabla f(x,\xi)=\nabla f(x)+\zeta$, где $\zeta$ — гауссов шум с
$\mathbb E\|\zeta\|^2=\sigma^2/b$ ($b$ — размер батча).

In [ ]:
class QuadraticProblem:
    '''f(x) = 0.5 x^T A x - b^T x, спектр A лог-равномерен на [mu, L].'''

    def __init__(self, d=50, mu=0.1, L=10.0, seed=0):
        rng = np.random.default_rng(seed)
        eig = np.exp(np.linspace(np.log(mu), np.log(L), d))
        Q, _ = np.linalg.qr(rng.standard_normal((d, d)))
        self.A = (Q * eig) @ Q.T
        self.eig = eig
        self.b = rng.standard_normal(d)
        self.xstar = np.linalg.solve(self.A, self.b)
        self.mu, self.L, self.d = mu, L, d

    def grad(self, X):
        '''X: (n_runs, d) -> (n_runs, d).'''
        return X @ self.A - self.b

    def f(self, X):
        return 0.5 * np.einsum("rd,de,re->r", X, self.A, X) - X @ self.b


def start_point(problem, R0, seed=0):
    '''Точка на расстоянии ровно R0 от x*.'''
    rng = np.random.default_rng(seed)
    u = rng.standard_normal(problem.d)
    return problem.xstar + R0 * u / np.linalg.norm(u)


PROB = QuadraticProblem(d=50, mu=0.1, L=10.0, seed=SEED)
SIGMA, R0 = 0.1, 5.0
X0 = start_point(PROB, R0, seed=SEED)          # ОДНА стартовая точка для Э1 и Э2
print(f"mu={PROB.mu}, L={PROB.L}, kappa={PROB.L / PROB.mu:.0f}, R0={R0}, sigma={SIGMA}")

---
# Экспериментальные задачи

Общее правило: везде, где в формуле стоит $\mathbb E$, в коде должно стоять усреднение по нескольким
независимым запускам (не по одной траектории!). Все графики — в логарифмическом масштабе по оси $y$
(а где указано — и по $x$).

## Эксперимент Э1 (10 баллов). Плато SGD: роль шага и размера батча

Рассмотрим квадратичную задачу с известными $\mu$ и $L$:
$$f(x)=\tfrac12 x^{\!\top}Ax-\langle b,x\rangle,\qquad A=A^{\!\top}\succ0,\quad \operatorname{spec}(A)\subset[\mu,L],$$
и стохастический оракул $\nabla f(x,\xi)=\nabla f(x)+\zeta$, где $\zeta$ — гауссов шум с
$\mathbb E\|\zeta\|^2=\sigma^2/b$ ($b$ — размер батча).

**Что нужно сделать.**

1. Реализовать SGD с постоянным шагом, векторизованно по `n_runs` независимым запускам.
2. Построить $\mathbb E\|x_k-x^*\|^2$ (усреднение по запускам) для $\gamma\in\{1/L,\ 1/(4L),\ 1/(16L)\}$
   вместе с теоретической границей $(1-\gamma\mu)^kR_0^2+\gamma\sigma^2/\mu$ из занятия 3.
3. Проверить численно, что уровень «плато» пропорционален $\gamma$: измерить плато как медиану
   $\|x_k-x^*\|^2$ по последней четверти итераций и вывести таблицу
   «$\gamma$ | измеренное плато | граница $\gamma\sigma^2/\mu$ | отношение».
4. Повторить для $b\in\{1,4,16\}$ при фиксированном $\gamma=1/L$ и проверить, что плато падает как $1/b$.
5. **Вывод:** сравнить два способа уменьшить плато в 4 раза — «уменьшить шаг в 4 раза» и
   «увеличить батч в 4 раза» — по числу *итераций* и по числу *обращений к оракулу*, нужных
   для выхода на плато.

In [ ]:
def run_sgd(problem, K, gamma, sigma, batch=1, n_runs=100, x0=None, seed=0):
    '''SGD с постоянным или переменным шагом.

    gamma  : число либо функция k -> gamma_k
    return : массив длины K+1 с оценкой E||x_k - x*||^2
    '''
    # TODO: инициализировать n_runs копий x0, на каждом шаге прибавлять к градиенту
    #       гауссов шум с E||zeta||^2 = sigma^2 / batch и записывать среднее по запускам
    raise NotImplementedError

> **Вывод Э1.** …

## Эксперимент Э2 (10 баллов). Расписания шага и скорость $1/K$

На той же квадратичной задаче (но с более крупным шумом $\sigma=1$ — иначе стохастический режим
не виден на доступных горизонтах) сравните расписания шага (все — при $\gamma_0=1/L$, если не сказано иное):

| Обозначение | Правило |
|---|---|
| `const 1/L` | $\gamma_k=1/L$ |
| `Stich(K)` | $\gamma=\min\{1/L,\ \ln(\max\{2,\mu^2R_0^2K/\sigma^2\})/(\mu K)\}$ |
| `1/k` | $\gamma_k=\min\{1/L,\ 2/(\mu(k+1))\}$ |
| `1/sqrt(k)` | $\gamma_k=\gamma_0/\sqrt{k+1}$ |
| `step-decay` | $\gamma_0$, делится пополам каждые $K/\lfloor\log_2K\rfloor$ итераций |
| `warmup+cosine` | линейный warm-up 5% итераций, затем $\gamma_k=\gamma_0\cdot\tfrac12(1+\cos(\pi t))$ |

**Откуда эти расписания (обязательно посмотреть первоисточники).**

| Расписание | Источник |
|---|---|
| $\gamma_k\propto1/k$ | [Robbins, Monro. *A Stochastic Approximation Method.* Ann. Math. Statist. 22(3), 1951](https://doi.org/10.1214/aoms/1177729586); неасимптотический разбор — [Moulines, Bach. NeurIPS 2011](https://hal.science/hal-00608041) |
| $\gamma_k\propto1/\sqrt k$ | [Zinkevich. *Online Convex Programming and Generalized Infinitesimal Gradient Ascent.* ICML 2003](https://cdn.aaai.org/ICML/2003/ICML03-120.pdf); [Nemirovski, Juditsky, Lan, Shapiro. *Robust Stochastic Approximation Approach to Stochastic Programming.* SIAM J. Optim. 19(4), 2009](https://doi.org/10.1137/070704277) |
| постоянный шаг «под горизонт» $K$ | [Stich. *Unified Optimal Analysis of the (Stochastic) Gradient Method.* arXiv:1907.04232, 2019](https://arxiv.org/abs/1907.04232) — именно этот тюнинг разбирался на занятии 3 |
| step-decay | [Ge, Kakade, Kidambi, Netrapalli. *The Step Decay Schedule.* NeurIPS 2019, arXiv:1904.12838](https://arxiv.org/abs/1904.12838) |
| warm-up | [Goyal et al. *Accurate, Large Minibatch SGD: Training ImageNet in 1 Hour.* arXiv:1706.02677, 2017](https://arxiv.org/abs/1706.02677) (раздел 2.2) |
| cosine annealing | [Loshchilov, Hutter. *SGDR: Stochastic Gradient Descent with Warm Restarts.* ICLR 2017, arXiv:1608.03983](https://arxiv.org/abs/1608.03983). В PyTorch — `CosineAnnealingLR` / `CosineAnnealingWarmRestarts` |
| сходимость по последней точке | [Shamir, Zhang. *SGD for Non-smooth Optimization: Convergence Results and Optimal Averaging Schemes.* ICML 2013, arXiv:1212.1824](https://arxiv.org/abs/1212.1824) |

**Что нужно сделать.**

1. Для каждого расписания и каждого горизонта $K$ из сетки (например `[100, 300, 1000, 3000, 10000]`)
   **запустить метод заново** (расписания зависят от $K$!) и записать $\mathbb E\|x_K-x^*\|^2$.
2. Построить график «$\mathbb E\|x_K-x^*\|^2$ от $K$» через `plot_sweep` и добавить эталон
   $\sigma^2/(\mu^2K)$.
3. Оценить эмпирический наклон каждой кривой (линейная регрессия по $\log K$ и $\log$ значения) — отдельно
   по всей сетке и по трём наибольшим $K$ (на малых $K$ метод ещё в детерминированной фазе,
   и наклон там не про $\sigma^2/(\mu^2K)$) — и вывести таблицу.
4. **Вывод:** какие расписания дают наклон $\approx-1$, а какие нет; насколько «инженерные»
   `step-decay` и `warmup+cosine` близки к теоретически оптимальному `Stich(K)`;
   что происходит с `const 1/L` и почему.

In [ ]:
# TODO: словарь расписаний вида {name: (K -> (k -> gamma_k))}, прогон по сетке K, график и таблица наклонов

> **Вывод Э2.** …

## Эксперимент Э3 (10 баллов). Субградиентный метод на негладкой задаче

Задача робастной регрессии:
$$f(x)=\frac1m\|Ax-b\|_1\ \longrightarrow\ \min_{x\in\mathbb R^d},$$
$b$ содержит выбросы. Функция выпукла, $M$-липшицева, **не** гладкая, $f^*$ считается точно как
линейная программа (`scipy.optimize.linprog`).

**Что нужно сделать.**

1. Посчитать $f^*$ и $x^*$ через LP, взять $x_0=0$, $R_0=\|x_0-x^*\|$.
2. Оценить константу липшицевости двумя способами: теоретической верхней границей
   $M\le\|A\|_2/\sqrt m$ (докажите это!) и эмпирически — максимумом $\|g_k\|$ вдоль траектории.
3. Реализовать и сравнить:
   * постоянный шаг $\gamma=R_0/(M\sqrt K)$;
   * $\gamma_k=R_0/(M\sqrt{k+1})$;
   * шаг Поляка $\gamma_k=(f(x_k)-f^*)/\|g_k\|^2$;
   * AdaGrad-Norm $\gamma_k=D/b_k$, $D=R_0$.
4. Построить $f(x_k)-f^*$ (и «лучшее значение до момента $k$») через `plot_curves`, добавить саму
   теоретическую границу $MR_0/\sqrt k$ из занятия 4. Отдельно показать, что $f(x_k)$ **не монотонна**.
   Сравните наблюдаемый наклон с теоретическим $-1/2$; если он оказался круче — объясните, почему
   оценка $O(MR_0/\sqrt K)$ не обязана быть точной **на конкретной задаче** (подсказка: $f$ кусочно-линейна,
   а у полиэдральных функций минимум «острый»).
5. Проверить устойчивость к неверной калибровке: постоянный шаг с $M\to10M$ и $M\to M/10$,
   AdaGrad-Norm с $D\to10D$ и $D\to D/10$.
6. **Вывод:** выполняется ли граница $MR_0/\sqrt K$ и насколько она туга; как выглядит деградация
   при завышенном и при заниженном шаге (она несимметрична!); кто устойчивее к ошибке в константах.

In [ ]:
from scipy.optimize import linprog


def make_l1_problem(m=200, d=40, n_outliers=20, seed=7):
    rng = np.random.default_rng(seed)
    A = rng.standard_normal((m, d))
    x_true = rng.standard_normal(d)
    b = A @ x_true + 0.1 * rng.standard_normal(m)
    idx = rng.choice(m, size=n_outliers, replace=False)
    b[idx] += 10.0 * rng.standard_normal(n_outliers)
    return A, b


A_l1, b_l1 = make_l1_problem()
M_, D_ = A_l1.shape


def f_l1(x):
    return np.abs(A_l1 @ x - b_l1).sum() / M_


def subgrad_l1(x):
    return A_l1.T @ np.sign(A_l1 @ x - b_l1) / M_


# f* и x* через ЛП:  min (1/m) sum t   s.t.  -t <= Ax - b <= t
c = np.concatenate([np.zeros(D_), np.ones(M_) / M_])
A_ub = np.block([[A_l1, -np.eye(M_)], [-A_l1, -np.eye(M_)]])
b_ub = np.concatenate([b_l1, -b_l1])
res = linprog(c, A_ub=A_ub, b_ub=b_ub,
              bounds=[(None, None)] * D_ + [(0, None)] * M_, method="highs")
assert res.status == 0, res.message
x_star_l1, f_star_l1 = res.x[:D_], res.fun
print(f"f* = {f_star_l1:.6f}")

In [ ]:
# TODO: реализовать общий прогон субградиентного метода с произвольным правилом шага
#       и сравнить четыре правила + мисспецификацию констант

> **Вывод Э3.** …

## Эксперимент Э4 (10 баллов). Адаптивные методы; расходимость Adam

### (а) Логистическая регрессия с плохо масштабированными признаками

$$f(w)=\frac1n\sum_{i=1}^n\log\big(1+e^{-y_i\langle x_i,w\rangle}\big)+\frac\lambda2\|w\|^2 ,$$
где $j$-й признак умножен на масштаб $s_j$, $s_j$ пробегает $[10^{-2},10^{2}]$ логарифмически
(это делает задачу плохо обусловленной **покоординатно** — как раз тот случай, ради которого
придуманы адаптивные методы).

**Что нужно сделать.**

1. Реализовать (вручную, без готовых оптимизаторов) мини-батчевые **SGD, AdaGrad, RMSProp, Adam, AdamW**
   по формулам из лекции.
2. Для каждого метода подобрать шаг по сетке (например `np.logspace(-3, 0.5, 8)`), нарисовать
   лучшую кривую $f(w_t)-f^*$ по эпохам ($f^*$ посчитать L-BFGS'ом) и отдельным графиком —
   зависимость финального $f-f^*$ **от шага** (чувствительность к настройке).
   Сетка шагов должна накрывать оптимум **изнутри**: если лучший шаг оказался на краю сетки,
   сетку надо расширить.
3. Для AdamW сравнить два способа регуляризации: $\ell_2$ **внутри** функции потерь (Adam)
   и **decoupled** weight decay $x_{k+1}=(1-\lambda_k)x_k-\frac{D}{b_k}m_k$ (AdamW).
4. **Вывод:** кто выигрывает и почему; насколько шире «рабочий диапазон» шага у адаптивных методов.

### (б) Контрпример Reddi: Adam может расходиться

Рассмотрим одномерную онлайн-задачу на отрезке $[-1,1]$ с периодической последовательностью
$$f_t(x)=\begin{cases}Cx,& t\equiv1\ (\mathrm{mod}\ 3)\\ -x,&\text{иначе}\end{cases}\qquad C>2 .$$
Средняя функция равна $\frac{C-2}{3}x$, поэтому оптимум — $x^*=-1$.

**Первоисточники методов.**

| Метод | Источник |
|---|---|
| AdaGrad | [Duchi, Hazan, Singer. *Adaptive Subgradient Methods for Online Learning and Stochastic Optimization.* JMLR 12, 2011](https://www.jmlr.org/papers/v12/duchi11a.html) |
| AdaGrad-Norm | [Streeter, McMahan. *Less Regret via Online Conditioning.* arXiv:1002.4908, 2010](https://arxiv.org/abs/1002.4908); анализ — [Ward, Wu, Bottou. *AdaGrad stepsizes: sharp convergence over nonconvex landscapes.* ICML 2019](https://arxiv.org/abs/1806.01811) |
| RMSProp | [Tieleman, Hinton. Coursera «Neural Networks for Machine Learning», лекция 6.5, 2012](https://www.cs.toronto.edu/~tijmen/csc321/slides/lecture_slides_lec6.pdf) |
| Adam | [Kingma, Ba. *Adam: A Method for Stochastic Optimization.* ICLR 2015, arXiv:1412.6980](https://arxiv.org/abs/1412.6980) |
| AMSGrad | [Reddi, Kale, Kumar. *On the Convergence of Adam and Beyond.* ICLR 2018 (best paper)](https://openreview.net/forum?id=ryQu7f-RZ), [arXiv:1904.09237](https://arxiv.org/abs/1904.09237) — оттуда и контрпример ниже |
| AdamW | [Loshchilov, Hutter. *Decoupled Weight Decay Regularization.* ICLR 2019, arXiv:1711.05101](https://arxiv.org/abs/1711.05101) |
| «что на самом деле делает Adam» | [Balles, Hennig. *Dissecting Adam: The Sign, Magnitude and Variance of Stochastic Gradients.* ICML 2018](https://arxiv.org/abs/1705.07774) |

**Что нужно сделать.** Реализовать Adam и AMSGrad ($\hat b_k^2=\max\{\hat b_{k-1}^2,b_k^2\}$) с проекцией
на $[-1,1]$ и шагом $\alpha_t=\alpha/\sqrt t$; прогнать сетку по $C$ и $\beta_2$ и вывести таблицу
финальных $x_T$. **Вывод:** к чему сходится Adam, к чему AMSGrad, и в чём причина
(подсказка: знак «эффективного» шага $\Gamma_t=\frac{\sqrt{v_t}}{\alpha_t}-\frac{\sqrt{v_{t-1}}}{\alpha_{t-1}}$).

In [ ]:
def make_logreg(n=4000, d=100, lam=1e-3, seed=11):
    rng = np.random.default_rng(seed)
    scales = np.logspace(-2, 2, d)
    X = rng.standard_normal((n, d)) * scales
    w_true = rng.standard_normal(d) / (np.sqrt(d) * scales)
    p = 1.0 / (1.0 + np.exp(-X @ w_true))
    y = np.where(rng.random(n) < p, 1.0, -1.0)
    return X, y, lam


X_lr, y_lr, LAM = make_logreg()
N_LR, D_LR = X_lr.shape


def f_logreg(w, with_reg=True):
    z = -y_lr * (X_lr @ w)
    val = np.mean(np.logaddexp(0.0, z))
    return val + (LAM / 2) * w @ w if with_reg else val


def grad_logreg(w, idx=None, with_reg=True):
    Xb = X_lr if idx is None else X_lr[idx]
    yb = y_lr if idx is None else y_lr[idx]
    z = -yb * (Xb @ w)
    s = np.exp(-np.logaddexp(0.0, -z))    # sigma(z) = sigma(-y<x,w>), устойчиво
    g = -(Xb.T @ (yb * s)) / len(yb)
    return g + LAM * w if with_reg else g

In [ ]:
# TODO: единый цикл обучения + функции обновления SGD / AdaGrad / RMSProp / Adam / AdamW

> **Вывод Э4.** …

## Эксперимент Э5 (10 баллов). Parameter-free методы: цена незнания $D=\|x^0-x^*\|$

По материалу занятия 5. Напомним логику лекции:

* **Baseline.** Субградиентный спуск с $\gamma=\dfrac{\|x^0-x^*\|}{M\sqrt T}$ даёт
  $f(\bar x^T)-f^*=O\!\big(\tfrac{M\|x^0-x^*\|}{\sqrt T}\big)$ — но требует знания **двух** констант.
* **AdaGrad-Norm**, $\gamma^t=\dfrac{1}{\sqrt{\sum_{i\le t}\|g^i\|^2}}$, снимает зависимость от $M$,
  но не от $D=\|x^0-x^*\|$.
* **Наивный подход — Bisection.** Идеальный постоянный шаг
  $\gamma_{\mathrm{opt}}=\dfrac{\max_{0\le i\le T-1}\|x^0-x^i\|}{\sqrt{\sum_{i=0}^{T-1}\|g^i\|^2}}$
  зависит от самого себя; ищем неподвижную точку $\gamma=\phi(\gamma)$ бисекцией по $\log\gamma$
  за $O\big(\log\log\frac{\gamma_{hi}}{\gamma_{lo}}\big)$ запусков оптимизатора.
* **DoG.** $\gamma^t=\dfrac{\max_{0\le i\le t}\|x^0-x^i\|}{\sqrt{\alpha\sum_{i\le t}\|g^i\|^2}}$ —
  тот же шаг, но по уже известной на текущий момент информации.
* **D-Adaptation** (Алгоритм 2 из лекции) — оценка $d^t$, **ограниченная сверху** самим $D$ и растущая к нему;
  **Prodigy** — та же схема с шагом $\lambda^t=\dfrac{(d^t)^2}{\sqrt{\sum_{i\le t}(d^i)^2\|g^i\|^2}}$.

**Первоисточники**

| Тема | Источник (в квадратных скобках — номер в списке литературы слайдов) |
|---|---|
| Bisection, «наивный» parameter-free | [Carmon, Hinder. *Making SGD Parameter-Free.* COLT 2022, arXiv:2205.02160](https://arxiv.org/abs/2205.02160) |
| DoG | [Ivgi, Hinder, Carmon. *DoG is SGD's Best Friend.* ICML 2023, arXiv:2302.12022](https://arxiv.org/abs/2302.12022) |
| D-Adaptation | [Defazio, Mishchenko. *Learning-Rate-Free Learning by D-Adaptation.* ICML 2023, arXiv:2301.07733](https://arxiv.org/abs/2301.07733) |
| Prodigy | [Mishchenko, Defazio. *Prodigy: An Expeditiously Adaptive Parameter-Free Learner.* arXiv:2306.06101, 2023](https://arxiv.org/abs/2306.06101) |
| sign-SGD | [Bernstein, Wang, Azizzadenesheli, Anandkumar. *signSGD: Compressed Optimisation for Non-Convex Problems.* ICML 2018, arXiv:1802.04434](https://arxiv.org/abs/1802.04434) |
| контрпримеры для sign-SGD | [Karimireddy, Rebjock, Stich, Jaggi. *Error Feedback Fixes SignSGD and other Gradient Compression Schemes.* ICML 2019, arXiv:1901.09847](https://arxiv.org/abs/1901.09847) |
| геометрия sign-спуска, связь $L_\infty$ и $L_2$ | [Balles, Pedregosa, Le Roux. *The Geometry of Sign Gradient Descent.* arXiv:2002.08056, 2020](https://arxiv.org/abs/2002.08056) |
| ALIAS Sign-SGD | [Medyakov et al. *Sign-SGD via Parameter-Free Optimization*](https://proceedings.iclr.cc/paper_files/paper/2026/file/1199e1805888fabf6edff851ab86c057-Paper-Conference.pdf) |
| бенчмарк, «насколько мы близки к hyperparameter-free» | [Dahl et al. *Benchmarking Neural Network Training Algorithms* (AlgoPerf), arXiv:2306.07179](https://arxiv.org/abs/2306.07179); [Kasimbeg et al. arXiv:2505.24005, 2025](https://arxiv.org/abs/2505.24005) |

**Полигон:** негладкая задача из Э3, $f(x)=\frac1m\|Ax-b\|_1$, для которой $x^*$ и $f^*$ известны точно
(ЛП), значит известно и «оракульное» $D=\|x^0-x^*\|$ — именно то число, которое parameter-free методы
не имеют права использовать.

**Что нужно сделать.**

**(а) Цена незнания $D$.** Запустите (i) субградиентный метод с постоянным шагом
$\gamma=\dfrac{D_{\text{guess}}}{M\sqrt T}$ и (ii) AdaGrad-Norm с $\gamma^t=\dfrac{D_{\text{guess}}}{\sqrt{\sum_{i\le t}\|g^i\|^2}}$
для $D_{\text{guess}}/D$ от $10^{-3}$ до $10^{3}$. Постройте «U-образную» кривую
«итоговое $f-f^*$ против $D_{\text{guess}}/D$». Насколько велика цена ошибки в $D$ на порядок
в обе стороны? Симметрична ли она?

**(б) Bisection.** Реализуйте $\phi(\gamma)$ и Алгоритм 1 из лекции. Постройте график $\phi(\gamma)$
вместе с диагональю $\gamma\mapsto\gamma$ (та самая картинка с занятия) и отметьте найденную точку.
Посчитайте **число запусков оптимизатора** и проверьте эмпирически зависимость
$O\big(\log\log\frac{\gamma_{hi}}{\gamma_{lo}}\big)$, меняя ширину исходной вилки на много порядков.
Сравните найденный $\gamma^*$ с оракульным $D/(M\sqrt T)$ и с лучшим шагом из сетки в (а).

**(в) DoG, D-Adaptation, Prodigy.** Реализуйте три метода по формулам лекции и:
1. сравните их с оракульным шагом и с bisection по кривой $\min_{t\le k}f(x^t)-f^*$;
2. проверьте **робастность**: прогоните их со стартовым приближением
   ($r_\varepsilon$ для DoG, $d^0$ для D-Adaptation/Prodigy) по **той же сетке
   $10^{-3}D\dots10^{3}D$**, что и в (а), и нанесите результат на тот же график;
3. постройте траекторию $d^t/D$ и проверьте, что для D-Adaptation выполняется $d^t\le D$
   (это и есть «ограниченная сверху аппроксимация» из лекции).

   Ответьте заодно на вопрос со слайда: **что такое $\alpha$ в DoG, зачем она и где здесь parameter-free?**

**(г) Sign-SGD (вторая половина лекции).** Воспроизведите контрпример:
$$\min_{x\in[-1,1]}f(x)=\frac14x=\frac14\big(4x-x-x-x\big),\qquad
g=\begin{cases}4,&\text{с вер. }1/4\\-1,&\text{с вер. }3/4\end{cases}$$
Проверьте численно, что $\mathbb E_t[f(x^{t+1})]=f(x^t)+\gamma/8$, то есть sign-SGD с батчем 1
**расходится** при любом постоянном шаге, хотя $\mathbb E[g]=1/4>0$. Покажите, что переход к батчу
(знак **среднего** градиента) чинит направление. Объясните, почему из-за этого в лекции для sign-SGD
критерий формулируется через норму градиента, а parameter-free версия (ALIAS Sign-SGD) адаптируется
к $\Delta^*=f(x^0)-f(x^*)$ и $L_\infty$, а не к $D$ и $M$.

In [ ]:
# TODO (а): sg_const(gamma) и sg_adagrad_norm(D_guess); свип по D_guess/D в [1e-3, 1e3]
# TODO (б): probe(gamma) -> (N_T, D_T); phi = N_T/D_T; bisection(gamma_lo, gamma_hi) по Алгоритму 1
# TODO (в): dog(r_eps), d_adaptation(d0, prodigy=False/True) по Алгоритму 2
# TODO (г): игрушечный пример sign-SGD с батчем 1 / 8 / 64

> **Вывод Э5.** …

---

### Что стоит проверить перед сдачей

1. Ноутбук выполняется целиком (`Kernel → Restart & Run All`) без ошибок.
2. В каждом эксперименте есть письменный вывод, и он **сопоставляет** численный результат
   с теоретической оценкой из лекции: Э1 и Э2 — с теоремой о сходимости SGD и тюнингом шага
   (занятие 3), Э3 и Э4 — с оценкой $O(MR_0/\sqrt K)$ и адаптивными методами (занятие 4),
   Э5 — с parameter-free методами (занятие 5).
3. Графики построены **только** через `plot_curves` / `plot_sweep`: у всех кривых внутри одного
   эксперимента одна стартовая точка, подписаны оси и легенда.
4. Числовые сводки выведены через `show_table`.
5. Все запуски воспроизводимы (зафиксированы `seed`).